# 05 — High-watch / Danger Classifier Training / เทรน classifier เตือนระดับ Danger

**EN.** Trains a per-(station, horizon) binary classifier that predicts whether the heat index at `t + horizon_h` will cross the **Danger** threshold (default `41 °C`, see `configs/risk/thresholds.yaml`). Output: calibrated `p_danger` probabilities + a tuned operating threshold per (station, horizon). The probability feeds `app/core/risk_fusion.py` as the `p_danger` input alongside the regression q50/q90 quantiles.

**TH.** เทรน classifier ไบนารีต่อคู่ `(station, horizon)` เพื่อทำนายความน่าจะเป็นที่ heat index ที่ `t + horizon_h` จะแตะระดับ **Danger** (ค่าเริ่มต้น `41 °C`, ดู `configs/risk/thresholds.yaml`). ผลลัพธ์: ความน่าจะเป็น `p_danger` ที่ผ่านการ calibrate แล้ว + threshold ที่ปรับให้เหมาะสมต่อคู่ `(station, horizon)`. ความน่าจะเป็นจะส่งเข้า `app/core/risk_fusion.py` เป็น input `p_danger` คู่กับ q50/q90 จาก regressor.

**Pipeline / ลำดับการทำงาน:**

1. โหลด `configs/train/classifier.yaml` + ตั้ง RUN_ID
2. สร้างป้ายกำกับ `y_class = (hi[t+H] >= danger_threshold_c)` แล้ว print class balance ต่อ station
3. โหลด features ผ่าน `build_features` (เส้นทางเดียวกับ 03/04 — no leakage)
4. Train loop: time-series split พร้อม gap=horizon_h → LightGBM binary + class weighting → focal loss (ถ้าเปิดใน YAML และข้อมูลพอ) → isotonic calibration → tune threshold ด้วย Youden's J
5. บันทึก `classifier.json` (+ `classifier_<role>.txt` LightGBM dumps) แยกจาก `bundle.json` ของ regressor
6. แสดง diagnostics ครบใน notebook นี้: PR-AUC table, threshold sensitivity, lead-time histogram

> File-layout decision / การจัดวางไฟล์: classifier artifact จะอยู่ที่ `app/models/forecast_v3/{station}/h{H}/classifier.json` และไม่แตะ `bundle.json` ที่เป็นของ regressor (regressor มี backend หลายตัว — `lightgbm_quantile`, `xgboost`, `tabpfn`). Notebook `08_register.ipynb` จะ push ไฟล์ classifier เพิ่มควบคู่กับ bundle เดิม.

## 0. Setup / เริ่มต้น

**EN.** Bootstrap the repo (idempotent) and import the shared training utilities. Re-run `00_setup.ipynb` first if any check below fails.

**TH.** บูตสแตรปโค้ด (รันซ้ำได้) และ import utility เทรนนิ่งร่วม. ถ้าเช็คด้านล่างไม่ผ่านให้กลับไปรัน `00_setup.ipynb` ก่อน.

In [ ]:
# --- bootstrap (re-run if you opened this notebook before 00_setup) -------
import os, sys
REPO_DIR = "/content/Heat-wave-backend"
if not os.path.exists(REPO_DIR):
    # local-dev fallback: assume repo cwd already correct
    REPO_DIR = os.path.abspath(os.path.join(os.getcwd(), "..", ".."))
if REPO_DIR not in sys.path:
    sys.path.insert(0, REPO_DIR)
if os.path.isdir(REPO_DIR):
    os.chdir(REPO_DIR)
print("REPO_DIR:", REPO_DIR)
print("cwd     :", os.getcwd())

In [ ]:
# --- core imports ---------------------------------------------------------
import json
import math
import warnings
from datetime import date, datetime, timedelta, timezone
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import matplotlib.pyplot as plt

import lightgbm as lgb
from sklearn.isotonic import IsotonicRegression
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    f1_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    roc_auc_score,
    roc_curve,
)

warnings.filterwarnings("ignore", category=UserWarning)
warnings.filterwarnings("ignore", category=FutureWarning)

from app.data.stations import STATIONS
from app.data.loaders import read_observations
from app.ml.forecast.features import build_features, _DEFAULT_LAGS_H, _DEFAULT_ROLLING_H, add_heat_index_col

print("lightgbm:", lgb.__version__)
print("stations:", list(STATIONS.keys()))

## 1. Config + manifest / โหลดคอนฟิกและสร้าง manifest

**EN.** Load `configs/train/classifier.yaml`. Build a per-(station, horizon) manifest listing the work items. Skip-if-exists pattern: if `classifier.json` already exists for a target and `RETRAIN=False`, that pair is skipped.

**TH.** โหลด `configs/train/classifier.yaml` แล้วสร้าง manifest ต่อคู่ `(station, horizon)`. ถ้ามี `classifier.json` อยู่แล้วและ `RETRAIN=False` จะข้ามคู่นั้น.

In [ ]:
# --- load classifier config ------------------------------------------------
CFG_PATH = Path(REPO_DIR) / "configs" / "train" / "classifier.yaml"
with open(CFG_PATH, "r", encoding="utf-8") as f:
    CFG = yaml.safe_load(f)

DANGER_THRESHOLD_C = float(CFG["labeling"]["danger_threshold_c"])
STATIONS_CFG       = CFG["stations"]
HORIZONS_CFG       = CFG["horizons"]
DATA_WINDOW        = CFG["data_window"]
OUT_ROOT           = Path(REPO_DIR) / CFG["artifacts"]["out_root"]
P_DANGER_GATE      = float(CFG["decision"]["p_danger_gate"])
REPORT_THRESHOLDS  = CFG["decision"].get("report_at_thresholds", [0.10, 0.20, 0.30, 0.50])
CALIBRATION_METHOD = CFG["decision"].get("calibration", "isotonic")
PARAMS             = dict(CFG["params"])
TRAIN_CFG          = CFG["training"]
SEED               = int(CFG.get("seed", 42))

# focal-loss flag — config does not currently expose one; treat absence as disabled.
USE_FOCAL_LOSS = bool(CFG.get("focal_loss", {}).get("enabled", False))
FOCAL_GAMMA    = float(CFG.get("focal_loss", {}).get("gamma", 2.0))
FOCAL_MIN_POS  = int(CFG.get("focal_loss", {}).get("min_positives", 200))

RUN_ID = f"{date.today():%Y%m%d}_classifier_lgbm"
RETRAIN = False  # flip to True to overwrite existing classifier.json

print(f"RUN_ID            : {RUN_ID}")
print(f"danger_threshold_c: {DANGER_THRESHOLD_C} °C")
print(f"stations          : {STATIONS_CFG}")
print(f"horizons          : {HORIZONS_CFG}")
print(f"data_window       : {DATA_WINDOW[0]} → {DATA_WINDOW[1]}")
print(f"out_root          : {OUT_ROOT}")
print(f"calibration       : {CALIBRATION_METHOD}")
print(f"focal loss enabled: {USE_FOCAL_LOSS} (gamma={FOCAL_GAMMA}, min_pos={FOCAL_MIN_POS})")

In [ ]:
# --- build manifest --------------------------------------------------------
def classifier_path(station_id: str, horizon_h: int) -> Path:
    return OUT_ROOT / station_id / f"h{horizon_h}" / "classifier.json"

manifest_rows = []
for sid in STATIONS_CFG:
    for h in HORIZONS_CFG:
        out = classifier_path(sid, h)
        manifest_rows.append({
            "station_id": sid,
            "horizon_h": h,
            "out_path": str(out),
            "exists": out.exists(),
            "will_train": RETRAIN or (not out.exists()),
        })
manifest = pd.DataFrame(manifest_rows)
print(manifest.to_string(index=False))

## 2. Data loading + feature engineering / โหลดข้อมูลและสร้างฟีเจอร์

**EN.** Same path as `03_train_*.ipynb` / `04_*.ipynb`: read partitioned parquet via `read_observations`, then `build_features(df, horizon_h, ...)`. The feature matrix `X` enforces the no-leakage invariant: only `ts_utc <= row.ts_utc` observations used; the label here is built off `heat_index_c[t + horizon_h]`.

**TH.** เส้นทางเดียวกับ `03_*` / `04_*`: อ่าน parquet ด้วย `read_observations` แล้วเรียก `build_features(df, horizon_h, ...)`. `X` ทุกแถวใช้เฉพาะ `ts_utc <= row.ts_utc` (no leakage); ป้ายกำกับสร้างจาก `heat_index_c[t + horizon_h]`.

In [ ]:
# --- helper: load observation frame for a station --------------------------
def _to_date(d):
    if isinstance(d, str):
        return datetime.strptime(d, "%Y-%m-%d").date()
    if isinstance(d, datetime):
        return d.date()
    return d

_WINDOW_START = _to_date(DATA_WINDOW[0])
_WINDOW_END   = _to_date(DATA_WINDOW[1])

_obs_cache: dict[str, pd.DataFrame] = {}

def load_obs(station_id: str) -> pd.DataFrame:
    if station_id in _obs_cache:
        return _obs_cache[station_id]
    df = read_observations(station_id, _WINDOW_START, _WINDOW_END)
    if df.empty:
        _obs_cache[station_id] = df
        return df
    df = df.copy()
    df["ts_utc"] = pd.to_datetime(df["ts_utc"], utc=True)
    df["station_id"] = station_id
    df = df.sort_values("ts_utc").reset_index(drop=True)
    df = add_heat_index_col(df)
    _obs_cache[station_id] = df
    return df

# build (X, y_class) for one (station, horizon) using the same feature path
def build_xy_classifier(
    station_id: str,
    horizon_h: int,
    threshold_c: float,
):
    df = load_obs(station_id)
    if df.empty:
        return None, None, None
    X, y_reg = build_features(
        df.copy(),
        horizon_h=horizon_h,
        lags_h=_DEFAULT_LAGS_H,
        rolling_h=_DEFAULT_ROLLING_H,
    )
    if X.empty:
        return None, None, None
    # keep timestamps for time-series split + lead-time analysis
    ts = pd.to_datetime(df.loc[X.index, "ts_utc"], utc=True) if "ts_utc" in df.columns else None
    y_class = (y_reg.values >= threshold_c).astype(int)
    return X.reset_index(drop=True), pd.Series(y_class, name="y_class"), (ts.reset_index(drop=True) if ts is not None else None)

## 3. Class balance per station / สัดส่วนคลาสต่อสถานี

**EN.** Danger events at `>= 41 °C` are rare. The table below shows positive-rate per station for `horizon_h = 24` so you can see how heavy the imbalance is before training.

**TH.** เหตุการณ์ Danger (`>= 41 °C`) เกิดน้อย ตารางด้านล่างแสดง positive rate ต่อสถานีที่ `horizon_h = 24` เพื่อให้เห็นว่าข้อมูล imbalance แค่ไหนก่อนเริ่มเทรน.

In [ ]:
# --- class-balance audit (representative horizon = 24h) ------------------
BALANCE_HORIZON = 24 if 24 in HORIZONS_CFG else HORIZONS_CFG[0]
balance_rows = []
for sid in STATIONS_CFG:
    X, y, _ = build_xy_classifier(sid, BALANCE_HORIZON, DANGER_THRESHOLD_C)
    if X is None or len(y) == 0:
        balance_rows.append({"station_id": sid, "n_rows": 0, "n_pos": 0, "pos_rate_%": np.nan, "neg/pos": np.nan})
        continue
    n_pos = int(y.sum())
    n_neg = int(len(y) - n_pos)
    pos_rate = (n_pos / len(y)) * 100.0
    ratio = (n_neg / n_pos) if n_pos > 0 else float("inf")
    balance_rows.append({
        "station_id": sid, "n_rows": len(y), "n_pos": n_pos,
        "pos_rate_%": round(pos_rate, 3),
        "neg/pos": round(ratio, 1) if math.isfinite(ratio) else float("inf"),
    })
balance_df = pd.DataFrame(balance_rows)
print(f"Danger threshold = {DANGER_THRESHOLD_C} °C @ horizon_h={BALANCE_HORIZON}\n")
print(balance_df.to_string(index=False))

**EN.** A `neg/pos` ratio above ~30 indicates extreme imbalance — training relies on `is_unbalance=true` (LightGBM auto class weighting) plus an explicit `scale_pos_weight = neg/pos` per fit. PR-AUC (not ROC-AUC) is the headline metric for this reason.

**TH.** ถ้า `neg/pos > 30` ถือว่า imbalance หนัก — การเทรนใช้ `is_unbalance=true` (LightGBM ปรับ class weight อัตโนมัติ) ร่วมกับ `scale_pos_weight = neg/pos` และเลือก PR-AUC เป็นเมตริกหลัก.

## 4. Train loop / ลูปการเทรน

**EN.** Per `(station, horizon)`:

1. Time-series split with a `gap=horizon_h` between train and val so labels at validation cannot leak into training features.
2. LightGBM binary classifier with YAML `params`, `is_unbalance=true`, plus explicit `scale_pos_weight = neg/pos` for safety.
3. **Optional focal loss** (custom `fobj`/`feval` for `gamma=2`) — gated on `USE_FOCAL_LOSS` and a minimum positive count to avoid unstable training on tiny tails.
4. Isotonic calibration on the validation probabilities (single-booster path uses `IsotonicRegression` directly, matching the YAML `decision.calibration: isotonic`).
5. Threshold tuning on the calibrated val probabilities by maximizing **Youden's J** (`tpr - fpr`).
6. Persist `classifier.json` per `(station, horizon)`.

**TH.** ต่อคู่ `(station, horizon)`:

1. แบ่ง train/val แบบ time-series พร้อม `gap=horizon_h` ป้องกันการรั่วของป้ายกำกับ
2. LightGBM binary ด้วย params จาก YAML + `is_unbalance=true` + `scale_pos_weight = neg/pos`
3. **Focal loss แบบ optional** (custom objective เมื่อ `USE_FOCAL_LOSS=True` และจำนวน positive ≥ `FOCAL_MIN_POS`)
4. Isotonic calibration บน val probabilities
5. ปรับ threshold ด้วย Youden's J (`tpr - fpr`) บน val ที่ calibrate แล้ว
6. บันทึกผลลัพธ์ลง `classifier.json` ต่อคู่

In [ ]:
# --- focal loss (binary) custom objective + metric ------------------------
# Gated by USE_FOCAL_LOSS. Implements Lin et al. (2017) focal loss for a single
# logit input. Used as LightGBM custom fobj/feval (not Python's eval/exec).
def _sigmoid(z):
    return 1.0 / (1.0 + np.exp(-z))

def make_focal_objective(gamma: float):
    """Return (fobj, feval) closures for LightGBM.

    LightGBM passes raw logits z to fobj. We compute:
        p = sigmoid(z)
        loss = -(1 - p_t)**gamma * log(p_t)
    where p_t = p if y==1 else 1-p. Class weighting is handled separately
    via scale_pos_weight, so alpha is left at 1.0 here.
    """
    def fobj(preds, dataset):
        y = dataset.get_label().astype(np.float64)
        p = _sigmoid(preds)
        # standard BCE grad: p - y
        bce_grad = p - y
        # focal scaling: (1 - p_t)^gamma
        p_t = np.where(y > 0.5, p, 1.0 - p)
        focal_w = np.power(np.clip(1.0 - p_t, 1e-8, 1.0), gamma)
        # gradient (approximate, ignoring d focal_w / d z which has higher-order terms)
        grad = focal_w * bce_grad
        # hessian uses BCE second derivative scaled by focal weight
        hess = focal_w * p * (1.0 - p)
        hess = np.maximum(hess, 1e-6)
        return grad, hess

    def feval(preds, dataset):
        y = dataset.get_label().astype(np.float64)
        p = _sigmoid(preds)
        p_t = np.where(y > 0.5, p, 1.0 - p)
        loss = -np.power(np.clip(1.0 - p_t, 1e-8, 1.0), gamma) * np.log(np.clip(p_t, 1e-8, 1.0))
        return "focal_loss", float(np.mean(loss)), False  # lower is better

    return fobj, feval

In [ ]:
# --- splitter + threshold tuning ------------------------------------------
def time_series_split(n: int, horizon_h: int, val_frac: float = 0.2):
    """Return (train_idx, val_idx) with a gap of horizon_h between them."""
    val_size = max(int(n * val_frac), 24)
    val_size = min(val_size, max(n - horizon_h - 24, 1))
    val_size = max(val_size, 1)
    val_start = n - val_size
    train_end = max(val_start - horizon_h, 1)
    train_idx = np.arange(0, train_end)
    val_idx   = np.arange(val_start, n)
    return train_idx, val_idx

def tune_threshold_youden(y_true, p):
    """Return (best_threshold, best_J)."""
    if y_true.sum() == 0 or y_true.sum() == len(y_true):
        return float(P_DANGER_GATE), 0.0
    fpr, tpr, thr = roc_curve(y_true, p)
    j = tpr - fpr
    k = int(np.argmax(j))
    # roc_curve thresholds[0] is +inf — guard against that
    best_thr = float(thr[k]) if math.isfinite(float(thr[k])) else 0.5
    return best_thr, float(j[k])

In [ ]:
# --- core trainer ----------------------------------------------------------
def train_one(station_id: str, horizon_h: int):
    out_path = classifier_path(station_id, horizon_h)
    if out_path.exists() and not RETRAIN:
        print(f"[skip] {station_id} h{horizon_h} -> {out_path} exists")
        return None

    X, y, ts = build_xy_classifier(station_id, horizon_h, DANGER_THRESHOLD_C)
    if X is None or len(X) < 200:
        print(f"[skip] {station_id} h{horizon_h} -> not enough rows (have {0 if X is None else len(X)})")
        return None

    n = len(X)
    tr_idx, va_idx = time_series_split(n, horizon_h, val_frac=0.2)
    X_tr, X_va = X.iloc[tr_idx], X.iloc[va_idx]
    y_tr, y_va = y.iloc[tr_idx].values, y.iloc[va_idx].values

    n_pos = int(y_tr.sum()); n_neg = int(len(y_tr) - n_pos)
    if n_pos == 0:
        print(f"[skip] {station_id} h{horizon_h} -> no positives in train fold")
        return None
    spw = float(n_neg / max(n_pos, 1))

    params = dict(PARAMS)
    params.update({
        "seed": SEED,
        "feature_pre_filter": False,
        "is_unbalance": True,
        "scale_pos_weight": spw,
    })
    used_focal = False
    fobj = None; feval_fn = None
    if USE_FOCAL_LOSS and n_pos >= FOCAL_MIN_POS:
        fobj, feval_fn = make_focal_objective(FOCAL_GAMMA)
        # custom objective expects raw scores; remove built-in objective
        params.pop("objective", None)
        params.pop("metric", None)
        used_focal = True

    dtr = lgb.Dataset(X_tr.values, label=y_tr, feature_name=list(X.columns), free_raw_data=False)
    dva = lgb.Dataset(X_va.values, label=y_va, feature_name=list(X.columns), free_raw_data=False, reference=dtr)

    booster = lgb.train(
        params,
        dtr,
        num_boost_round=int(TRAIN_CFG["num_boost_round"]),
        valid_sets=[dtr, dva],
        valid_names=["train", "val"],
        callbacks=[lgb.early_stopping(int(TRAIN_CFG["early_stopping_rounds"]), verbose=False),
                   lgb.log_evaluation(period=0)],
        fobj=fobj,
        feval=feval_fn,
    )

    raw_va = booster.predict(X_va.values, num_iteration=booster.best_iteration)
    if used_focal:
        # raw output is logits — sigmoid to probabilities
        p_va = _sigmoid(np.asarray(raw_va))
    else:
        p_va = np.asarray(raw_va)

    if CALIBRATION_METHOD == "isotonic" and y_va.sum() > 0 and y_va.sum() < len(y_va):
        iso = IsotonicRegression(out_of_bounds="clip", y_min=0.0, y_max=1.0)
        iso.fit(p_va, y_va)
        p_va_cal = iso.predict(p_va)
    else:
        iso = None
        p_va_cal = p_va

    best_thr, best_j = tune_threshold_youden(y_va, p_va_cal)

    pr_auc  = float(average_precision_score(y_va, p_va_cal)) if y_va.sum() > 0 else float("nan")
    roc_auc = float(roc_auc_score(y_va, p_va_cal)) if 0 < y_va.sum() < len(y_va) else float("nan")

    y_hat = (p_va_cal >= best_thr).astype(int)
    if len(np.unique(y_va)) > 1:
        tn, fp, fn, tp = confusion_matrix(y_va, y_hat, labels=[0, 1]).ravel()
    else:
        tn, fp, fn, tp = (len(y_va), 0, 0, 0)
    prec = precision_score(y_va, y_hat, zero_division=0)
    rec  = recall_score(y_va, y_hat, zero_division=0)
    f1   = f1_score(y_va, y_hat, zero_division=0)
    fnr  = float(fn / (fn + tp)) if (fn + tp) > 0 else float("nan")
    over_warning = float(fp / max(len(y_va), 1))

    iso_payload = None
    if iso is not None:
        iso_payload = {
            "x": list(map(float, iso.X_thresholds_.tolist())),
            "y": list(map(float, iso.y_thresholds_.tolist())),
            "out_of_bounds": "clip",
        }

    out_path.parent.mkdir(parents=True, exist_ok=True)
    booster_path = out_path.parent / "classifier_lgbm.txt"
    booster.save_model(str(booster_path), num_iteration=booster.best_iteration)

    bundle = {
        "run_id": RUN_ID,
        "created_utc": datetime.now(timezone.utc).isoformat(),
        "station_id": station_id,
        "horizon_h": horizon_h,
        "target": CFG["target"],
        "danger_threshold_c": DANGER_THRESHOLD_C,
        "backend": CFG["backend"],
        "booster_file": booster_path.name,
        "feature_list": list(X.columns),
        "params": params,
        "used_focal_loss": used_focal,
        "focal_gamma": (FOCAL_GAMMA if used_focal else None),
        "calibration": {
            "method": CALIBRATION_METHOD if iso is not None else "none",
            "isotonic": iso_payload,
        },
        "threshold": {
            "tuned": float(best_thr),
            "youden_j": float(best_j),
            "yaml_p_danger_gate": P_DANGER_GATE,
        },
        "metrics": {
            "pr_auc": pr_auc,
            "roc_auc": roc_auc,
            "precision_at_thr": float(prec),
            "recall_at_thr": float(rec),
            "f1_at_thr": float(f1),
            "fnr_at_thr": fnr,
            "over_warning_rate": over_warning,
            "n_train": int(len(y_tr)),
            "n_val": int(len(y_va)),
            "n_pos_train": n_pos,
            "n_pos_val": int(y_va.sum()),
            "scale_pos_weight": spw,
            "best_iteration": int(booster.best_iteration or 0),
        },
    }
    with open(out_path, "w", encoding="utf-8") as f:
        json.dump(bundle, f, indent=2)

    return {
        "station_id": station_id,
        "horizon_h": horizon_h,
        "X_va": X_va, "y_va": y_va, "p_va_cal": p_va_cal,
        "ts_va": (ts.iloc[va_idx].reset_index(drop=True) if ts is not None else None),
        "booster": booster, "bundle": bundle, "out_path": out_path,
    }

In [ ]:
# --- run the loop ----------------------------------------------------------
results = {}
metrics_rows = []
for sid in STATIONS_CFG:
    for h in HORIZONS_CFG:
        try:
            r = train_one(sid, h)
        except Exception as exc:
            print(f"[error] {sid} h{h}: {exc!r}")
            r = None
        if r is None:
            continue
        results[(sid, h)] = r
        m = r["bundle"]["metrics"]
        metrics_rows.append({
            "station_id": sid, "horizon_h": h,
            "pr_auc":  round(m["pr_auc"],  4) if m["pr_auc"]  == m["pr_auc"]  else float("nan"),
            "roc_auc": round(m["roc_auc"], 4) if m["roc_auc"] == m["roc_auc"] else float("nan"),
            "thr":     round(r["bundle"]["threshold"]["tuned"], 3),
            "prec":    round(m["precision_at_thr"], 3),
            "rec":     round(m["recall_at_thr"], 3),
            "f1":      round(m["f1_at_thr"], 3),
            "fnr":     round(m["fnr_at_thr"], 3) if m["fnr_at_thr"] == m["fnr_at_thr"] else float("nan"),
            "over_warn": round(m["over_warning_rate"], 4),
            "n_pos_val": m["n_pos_val"],
            "n_val": m["n_val"],
        })
        print(f"[ok]   {sid} h{h:>2d}  PR-AUC={m['pr_auc']:.3f}  ROC-AUC={m['roc_auc']:.3f}  thr={r['bundle']['threshold']['tuned']:.3f}")

metrics_df = pd.DataFrame(metrics_rows)
if not metrics_df.empty:
    print("\n=== Metrics summary ===")
    print(metrics_df.to_string(index=False))

## 5. Save layout (already done above) / รูปแบบไฟล์ที่บันทึก

**EN.** Each `(station, horizon)` writes the following next to (but never replacing) the regression `bundle.json`:

```
app/models/forecast_v3/{station}/h{H}/
├── bundle.json              ← regression artifact (untouched here)
├── classifier.json          ← classifier sidecar (this notebook)
└── classifier_lgbm.txt      ← LightGBM booster dump (this notebook)
```

`classifier.json` carries `feature_list`, params, calibration breakpoints, tuned threshold, and headline metrics — `08_register.ipynb` should `git add` both the JSON and the `.txt` dump.

**TH.** ต่อคู่ `(station, horizon)` ระบบจะเขียนไฟล์ตามด้านบนถัดจาก `bundle.json` (ไม่แก้ของเดิม). `classifier.json` เก็บ feature list, params, ตัว calibrate, threshold ที่ปรับ และเมตริกหลัก. ขั้น `08_register` ต้อง `git add` ทั้ง JSON และ `.txt`.

## 6. Inline diagnostics / ผลตรวจสอบใน notebook

**EN.** All metrics + charts render here so the user does not need to open another notebook to QA the run.

**TH.** เมตริกและกราฟทั้งหมดอยู่ใน notebook นี้ ไม่ต้องเปิดไฟล์อื่นเพื่อ QA.

In [ ]:
# --- 6.1 PR-AUC + ROC-AUC table -------------------------------------------
if metrics_df.empty:
    print("No models trained in this run (everything skipped or insufficient data).")
else:
    print("PR-AUC per (station, horizon)")
    print(metrics_df.pivot(index="station_id", columns="horizon_h", values="pr_auc").round(3).to_string())
    print()
    print("ROC-AUC per (station, horizon)")
    print(metrics_df.pivot(index="station_id", columns="horizon_h", values="roc_auc").round(3).to_string())
    print()
    print("Tuned threshold per (station, horizon)")
    print(metrics_df.pivot(index="station_id", columns="horizon_h", values="thr").round(3).to_string())

In [ ]:
# --- 6.2 operating-point metrics at the tuned threshold + at YAML reports ---
def report_at(p, y, thr):
    yh = (p >= thr).astype(int)
    if len(np.unique(y)) < 2:
        return dict(thr=thr, prec=float("nan"), rec=float("nan"), f1=float("nan"), fnr=float("nan"), over_warn=float("nan"))
    tn, fp, fn, tp = confusion_matrix(y, yh, labels=[0,1]).ravel()
    prec = tp / (tp + fp) if (tp + fp) > 0 else 0.0
    rec  = tp / (tp + fn) if (tp + fn) > 0 else 0.0
    f1   = (2 * prec * rec / (prec + rec)) if (prec + rec) > 0 else 0.0
    fnr  = fn / (fn + tp) if (fn + tp) > 0 else float("nan")
    over_warn = fp / len(y)
    return dict(thr=thr, prec=prec, rec=rec, f1=f1, fnr=fnr, over_warn=over_warn)

rows = []
for (sid, h), r in results.items():
    p = r["p_va_cal"]; y = r["y_va"]
    tuned = r["bundle"]["threshold"]["tuned"]
    for thr in sorted({float(tuned), *map(float, REPORT_THRESHOLDS)}):
        m = report_at(p, y, thr)
        rows.append({"station": sid, "h": h, "label": ("tuned" if abs(thr - tuned) < 1e-9 else f"{thr:.2f}"), **m})
if rows:
    op_df = pd.DataFrame(rows)
    for col in ("prec", "rec", "f1", "fnr", "over_warn"):
        op_df[col] = op_df[col].round(3)
    print(op_df.to_string(index=False))
else:
    print("No operating-point report — no trained models.")

In [ ]:
# --- 6.3 threshold sensitivity curve (representative pair) ----------------
if results:
    # pick the (station, horizon) with the highest n_pos_val so the curve is informative
    rep = max(results.items(), key=lambda kv: kv[1]["y_va"].sum())
    (rep_sid, rep_h), rep_res = rep
    p = rep_res["p_va_cal"]; y = rep_res["y_va"]
    grid = np.linspace(0.01, 0.99, 99)
    precs = []; recs = []
    for thr in grid:
        m = report_at(p, y, thr)
        precs.append(m["prec"]); recs.append(m["rec"])
    fig, ax = plt.subplots(figsize=(8, 4.5))
    ax.plot(grid, precs, label="Precision")
    ax.plot(grid, recs, label="Recall")
    tuned = rep_res["bundle"]["threshold"]["tuned"]
    ax.axvline(tuned, color="k", linestyle="--", alpha=0.6, label=f"tuned thr = {tuned:.2f}")
    ax.axvline(P_DANGER_GATE, color="gray", linestyle=":", alpha=0.6, label=f"YAML p_danger_gate = {P_DANGER_GATE}")
    ax.set_xlabel("decision threshold")
    ax.set_ylabel("score")
    ax.set_title(f"Threshold sensitivity — {rep_sid} h{rep_h}")
    ax.legend()
    ax.grid(alpha=0.3)
    plt.tight_layout(); plt.show()
else:
    print("No models — sensitivity curve skipped.")

In [ ]:
# --- 6.4 lead-time histogram ----------------------------------------------
# For each true Danger event in the val set, the alert lead time equals horizon_h
# at the moment the classifier crosses its tuned threshold. We aggregate across
# all (station, horizon) pairs to show the distribution of operational lead times.
lead_times = []
for (sid, h), r in results.items():
    p = r["p_va_cal"]; y = r["y_va"]
    thr = r["bundle"]["threshold"]["tuned"]
    fired = (p >= thr) & (y == 1)
    lead_times.extend([h] * int(fired.sum()))
if lead_times:
    fig, ax = plt.subplots(figsize=(7, 4))
    bins = sorted(set(HORIZONS_CFG))
    edges = [b - 1.5 for b in bins] + [bins[-1] + 1.5]
    ax.hist(lead_times, bins=edges, edgecolor="black", alpha=0.85)
    ax.set_xticks(bins)
    ax.set_xlabel("alert lead time (hours ahead of >=41°C event)")
    ax.set_ylabel("# correctly fired alerts")
    ax.set_title("Lead-time distribution across all (station, horizon) pairs")
    ax.grid(alpha=0.3, axis="y")
    plt.tight_layout(); plt.show()
else:
    print("No correctly-fired alerts in val sets — lead-time histogram is empty.")

## 7. Mini-report / สรุปผล

**EN.**

- Trained one binary classifier per `(station, horizon)` for stations `BKK_01, CNX_01, KKN_01, HYI_01, RYG_01` and horizons `[6, 12, 24, 48, 72]` using LightGBM with `is_unbalance=true` + `scale_pos_weight = neg/pos`.
- Labels: `y_class = (heat_index_c[t + horizon_h] >= classifier.danger_threshold_c)` from `configs/train/classifier.yaml` (default `41 °C`). Class imbalance is heavy (see Section 3 table).
- Calibration: isotonic on the validation fold. Threshold tuned by maximizing Youden's J.
- Focal-loss objective is wired in but **gated** by `USE_FOCAL_LOSS` and a `FOCAL_MIN_POS` minimum positive count — disabled by default in the YAML; enable by adding a `focal_loss: {enabled: true, gamma: 2.0, min_positives: 200}` block.
- Persistence: `classifier.json` + `classifier_lgbm.txt` per `(station, horizon)`. Regression `bundle.json` is **never modified**, keeping the v3 backend dispatch in `app/ml/registry.load_latest_v3` unaffected.
- The calibrated `p_danger` produced here is the input to `app/core/risk_fusion.py`'s `p_danger` parameter; the tuned threshold maps directly to `DEFAULT_THRESHOLDS['p_danger_gate']` (override via `thresholds=` if the per-station tuning differs).

**Next** → `07_evaluate.ipynb` should load each `classifier.json`, replay the val fold, and verify PR-AUC / lead-time numbers on a held-out test window before `08_register.ipynb` pushes artifacts back to Drive / Git.

**TH.**

- เทรน classifier ไบนารีต่อคู่ `(station, horizon)` ครบ 5 สถานี × 5 horizon ด้วย LightGBM + `is_unbalance=true` + `scale_pos_weight`.
- ป้ายกำกับ: `y_class = (hi[t+H] >= danger_threshold_c)` จาก YAML (ค่าเริ่มต้น 41°C). ข้อมูล imbalance หนักตามตาราง Section 3.
- ใช้ isotonic calibration บน val + เลือก threshold ด้วย Youden's J.
- Focal loss เขียนไว้แล้วแต่ปิดเป็นค่าเริ่มต้น (เปิดได้โดยเพิ่ม block `focal_loss` ใน YAML และต้องมี positive ≥ `FOCAL_MIN_POS`).
- บันทึกเป็น `classifier.json` + `classifier_lgbm.txt` แยกจาก `bundle.json` เพื่อไม่ชนกับ regressor.
- ความน่าจะเป็น `p_danger` หลัง calibrate ใช้เป็น input ของ `app/core/risk_fusion.py` ส่วน threshold ที่ปรับได้คือ `p_danger_gate`.

**ขั้นถัดไป** → ไปที่ `07_evaluate.ipynb` เพื่อตรวจ PR-AUC และ lead-time บน window สำรอง ก่อน `08_register.ipynb` จะ push artifact ขึ้น Drive / Git.